# Student Dropout Prediction - Preprocessing

This notebook prepares the fixed MVP dataset used by the modeling notebook and app. It removes `Enrolled`, keeps only the 10 MVP features, and saves both numeric and model-ready processed datasets.

In [23]:
# Import libraries used for preprocessing.
from pathlib import Path
import json

import pandas as pd

## Load Data

Resolve project paths, load the raw dataset, and load the MVP feature scope from `feature_config.json`.

In [24]:
# Resolve project paths.
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"
MVP_NUMERIC_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mvp_features_numeric.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_PATH)
print("Feature config path:", FEATURE_CONFIG_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Numeric MVP data path:", MVP_NUMERIC_DATA_PATH)

Project root: e:\Projects\student-dropout-prediction-ml
Raw data path: e:\Projects\student-dropout-prediction-ml\data\raw\dataset.csv
Feature config path: e:\Projects\student-dropout-prediction-ml\app\feature_config.json
Processed data path: e:\Projects\student-dropout-prediction-ml\data\processed\processed.csv
Numeric MVP data path: e:\Projects\student-dropout-prediction-ml\data\processed\mvp_features_numeric.csv


In [25]:
# Load raw data and feature config.
df = pd.read_csv(DATA_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

candidate_mvp_features = feature_config["features"]

print("Raw dataset shape:", df.shape)
print("Target distribution:")
print(df["Target"].value_counts())
print("\nMVP features:", candidate_mvp_features)

Raw dataset shape: (4424, 35)
Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

MVP features: ['Marital status', 'Course', 'Previous qualification', "Mother's qualification", "Father's qualification", 'Displaced', 'Educational special needs', 'Gender', 'Age at enrollment', 'International']


## Filter to Final MVP Scope

The numeric MVP dataset keeps original encoded feature values. `Enrolled` is removed, while target labels remain readable (`Graduate` / `Dropout`) for inspection.

In [26]:
# Keep fixed MVP features and remove Enrolled.
selected_columns = candidate_mvp_features + ["Target"]
missing_columns = [col for col in selected_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_mvp_numeric = df[selected_columns].copy()
df_mvp_numeric = df_mvp_numeric[df_mvp_numeric["Target"] != "Enrolled"].copy()

print("Numeric MVP dataset shape:", df_mvp_numeric.shape)
print("Target distribution after removing Enrolled:")
print(df_mvp_numeric["Target"].value_counts())

df_mvp_numeric.head()

Numeric MVP dataset shape: (3630, 11)
Target distribution after removing Enrolled:
Target
Graduate    2209
Dropout     1421
Name: count, dtype: int64


,Marital status,Course,Previous qualification,Mother's qualification,Father's qualification,Displaced,Educational special needs,Gender,Age at enrollment,International,Target
0,1,2,1,13,10,1,0,1,20,0,Dropout
1,1,11,1,1,3,1,0,1,19,0,Graduate
2,1,5,1,22,27,1,0,1,19,0,Dropout
3,1,15,1,23,27,1,0,0,20,0,Graduate
4,2,3,1,22,28,0,0,0,45,0,Graduate


## Save Numeric and Processed Data

`mvp_features_numeric.csv` keeps readable target labels. `processed.csv` uses the same numeric features but encodes the target as `Graduate = 0`, `Dropout = 1` for modeling.

In [27]:
# Save numeric MVP dataset and model-ready processed dataset.
target_mapping = {
    "Graduate": 0,
    "Dropout": 1
}

df_processed = df_mvp_numeric.copy()
df_processed["Target"] = df_processed["Target"].map(target_mapping)

if df_processed["Target"].isna().any():
    raise ValueError("Target mapping created missing values.")

MVP_NUMERIC_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_mvp_numeric.to_csv(MVP_NUMERIC_DATA_PATH, index=False)
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

print("Saved numeric MVP dataset to:", MVP_NUMERIC_DATA_PATH)
print("Saved processed dataset to:", PROCESSED_DATA_PATH)
print("Processed dataset shape:", df_processed.shape)
print("Encoded target distribution:")
print(df_processed["Target"].value_counts().sort_index())

Saved numeric MVP dataset to: e:\Projects\student-dropout-prediction-ml\data\processed\mvp_features_numeric.csv
Saved processed dataset to: e:\Projects\student-dropout-prediction-ml\data\processed\processed.csv
Processed dataset shape: (3630, 11)
Encoded target distribution:
Target
0    2209
1    1421
Name: count, dtype: int64


## Modeling Preprocessing Design

The modeling notebook applies these transformations:

- Binary flags (`Displaced`, `Educational special needs`, `Gender`, `International`): passthrough because they are already 0/1.
- `Age at enrollment`: `RobustScaler` because EDA shows right skew and high-age outliers.
- `Marital status`, `Course`, `Previous qualification`: one-hot encoding because they are nominal with manageable cardinality.
- Parents' qualification features: target encoding because they have many sparse categories.
- Models use class balancing and threshold tuning because Dropout recall is the priority.